# Safe Reload — Checkpoint Management

Demonstrates the correct way to fully reprocess a source from scratch: deleting only the checkpoint causes duplicate rows, because the target table still holds the previously loaded data. A safe reload must clear the checkpoint AND the target table together.

Runs on an isolated checkpoint, schema location, and target table — separate from the main `netflix_titles_stream` pipeline.

## Configuration

In [0]:
%run ./lab3_00_config

In [0]:
pipeline_name = "lab3_reload"

source_path = f"{storage_root}/ingestion/netflix_stream/"
reload_checkpoint_path = f"{storage_root}/checkpoints/{pipeline_name}/"
reload_schema_location = f"{storage_root}/schema_location/{pipeline_name}/"
reload_table = f"{catalog}.{bronze_schema}.netflix_titles_stream_reload"

print("Source path:", source_path)
print("Reload checkpoint:", reload_checkpoint_path)
print("Reload schema location:", reload_schema_location)
print("Reload table:", reload_table)

## 1. Initial load — create the isolated table

In [0]:
from pyspark.sql.functions import col, current_timestamp, current_date

df_reload_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", reload_schema_location)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(source_path)
)

df_reload_stream = (
    df_reload_stream
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_load_date", current_date())
)

query = (
    df_reload_stream.writeStream
    .option("checkpointLocation", reload_checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(reload_table)
)

query.awaitTermination()

initial_count = spark.table(reload_table).count()
print("Initial load completed. Row count:", initial_count)

## 2. Unsafe reload — delete only the checkpoint

Deleting the checkpoint without touching the target table makes Auto Loader forget which files were already processed. It will reprocess everything from scratch and append it — duplicating every row already in the table.

In [0]:
dbutils.fs.rm(reload_checkpoint_path, recurse=True)
print("Checkpoint deleted:", reload_checkpoint_path)

## 3. Re-run the stream — observe duplication

In [0]:
query = (
    df_reload_stream.writeStream
    .option("checkpointLocation", reload_checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(reload_table)
)

query.awaitTermination()

after_unsafe_count = spark.table(reload_table).count()
duplicated_rows = after_unsafe_count - initial_count

print("Row count AFTER unsafe reload:", after_unsafe_count)
print("Duplicate rows introduced:", duplicated_rows)

## 4. Safe reload — delete checkpoint AND clear the target table

The correct way to fully reprocess a source: reset both the checkpoint (so Auto Loader forgets what it already read) and the target table (so there's nothing left to duplicate into).

In [0]:
dbutils.fs.rm(reload_checkpoint_path, recurse=True)
spark.sql(f"DROP TABLE IF EXISTS {reload_table}")

print("Checkpoint deleted:", reload_checkpoint_path)
print("Table dropped:", reload_table)

## 5. Re-run after safe reload — confirm no duplicates

In [0]:
query = (
    df_reload_stream.writeStream
    .option("checkpointLocation", reload_checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(reload_table)
)

query.awaitTermination()

after_safe_count = spark.table(reload_table).count()
print("Row count AFTER safe reload:", after_safe_count)

if after_safe_count == initial_count:
    print(f"✅ SAFE RELOAD CONFIRMED — row count matches original ({initial_count}), no duplicates")
else:
    print(f"❌ Unexpected row count: expected {initial_count}, got {after_safe_count}")

## Conclusion

Deleting only the streaming checkpoint is unsafe because Auto Loader loses its record of previously processed files while the existing target data remains unchanged. As a result, all source files are processed again and appended, creating duplicate records.

A safe full reload requires resetting the checkpoint and clearing the isolated target table together. After applying this procedure, the source was reprocessed successfully and the final row count returned to the original 8810 rows without duplicates.

The main production checkpoint and Bronze table were not modified during this experiment.